In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn.metrics import pairwise_distances

# Change Working Dir To the Project Working Dir (for Noam)

In [2]:
# change the dir to the grandparent directory of the current working directory

current_dir = Path(os.getcwd())
grandparent_dir = current_dir.parent.parent
os.chdir(grandparent_dir)

In [3]:
os.getcwd()  # Check the cwd has updated

'c:\\Users\\noams\\Python Projects\\Audio_processing_project'

# Classical Clustering

## Naive Clustering

In [4]:
raw_data = pd.read_csv(Path("tcav_per_sample_with_acc.csv"))

In [5]:
raw_data.head(13)

,path,concept_name,layer_name,positive_percentage,magnitude,cav_acc,true_label,predicted_label,predicted_probability
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_constant_thick,module3.blocks.0.conv2,0.0,-0.479544,0.956522,calm,calm,0.999167
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_flat_thick,module3.blocks.0.conv2,0.0,-2.054141,0.869565,calm,calm,0.999167
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thick,module3.blocks.0.conv2,0.0,-0.572685,0.782609,calm,calm,0.999167
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thin,module3.blocks.0.conv2,0.0,-1.140559,0.913043,calm,calm,0.999167
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_flat_thick,module3.blocks.0.conv2,0.0,-0.252294,0.695652,calm,calm,0.999167
5,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_steep_thick,module3.blocks.0.conv2,0.0,-1.292134,0.869565,calm,calm,0.999167
6,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_steep_thin,module3.blocks.0.conv2,0.0,-0.815764,0.826087,calm,calm,0.999167
7,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,short_constant_thick,module3.blocks.0.conv2,0.0,-0.217394,0.826087,calm,calm,0.999167
8,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,short_dropping_steep_thick,module3.blocks.0.conv2,0.0,-0.479893,0.739130,calm,calm,0.999167
9,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,short_dropping_steep_thin,module3.blocks.0.conv2,0.0,-0.958675,0.826087,calm,calm,0.999167


In [6]:
emotion_encoder, concept_encoder = LabelEncoder(), LabelEncoder()

In [7]:
data_matching_predictions = raw_data.where(raw_data["true_label"] == raw_data["predicted_label"])

In [8]:
data_matching_predictions.head()

,path,concept_name,layer_name,positive_percentage,magnitude,cav_acc,true_label,predicted_label,predicted_probability
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_constant_thick,module3.blocks.0.conv2,0.0,-0.479544,0.956522,calm,calm,0.999167
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_flat_thick,module3.blocks.0.conv2,0.0,-2.054141,0.869565,calm,calm,0.999167
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thick,module3.blocks.0.conv2,0.0,-0.572685,0.782609,calm,calm,0.999167
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thin,module3.blocks.0.conv2,0.0,-1.140559,0.913043,calm,calm,0.999167
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_flat_thick,module3.blocks.0.conv2,0.0,-0.252294,0.695652,calm,calm,0.999167


In [9]:
encoded_emotions = emotion_encoder.fit_transform(data_matching_predictions["true_label"])
#encoded_concepts = concept_encoder.fit_transform(data_matching_predictions["concept_name"])
encoded_emotions

array([1, 1, 1, ..., 2, 2, 2])

In [10]:
data_matching_predictions["true_label"] = encoded_emotions
#data_matching_predictions["concept_name"] = encoded_concepts
data_matching_predictions["true_label"]

0        1
1        1
2        1
3        1
4        1
        ..
17275    2
17276    2
17277    2
17278    2
17279    2
Name: true_label, Length: 17280, dtype: int32

In [11]:
data_matching_predictions

,path,concept_name,layer_name,positive_percentage,magnitude,cav_acc,true_label,predicted_label,predicted_probability
0,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_constant_thick,module3.blocks.0.conv2,0.0,-0.479544,0.956522,1,calm,0.999167
1,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_flat_thick,module3.blocks.0.conv2,0.0,-2.054141,0.869565,1,calm,0.999167
2,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thick,module3.blocks.0.conv2,0.0,-0.572685,0.782609,1,calm,0.999167
3,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_dropping_steep_thin,module3.blocks.0.conv2,0.0,-1.140559,0.913043,1,calm,0.999167
4,RAVDESS\original_data\Actor_23\03-01-02-02-02-...,long_rising_flat_thick,module3.blocks.0.conv2,0.0,-0.252294,0.695652,1,calm,0.999167
...,...,...,...,...,...,...,...,...,...
17275,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_constant_thick,module3.blocks.0.conv2,1.0,0.739878,0.826087,2,disgust,0.999277
17276,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_dropping_steep_thick,module3.blocks.0.conv2,1.0,0.700198,0.739130,2,disgust,0.999277
17277,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_dropping_steep_thin,module3.blocks.0.conv2,1.0,0.733818,0.826087,2,disgust,0.999277
17278,RAVDESS\original_data\Actor_03\03-01-07-02-01-...,short_rising_steep_thick,module3.blocks.0.conv2,1.0,1.567698,0.956522,2,disgust,0.999277


In [12]:
vector_df = data_matching_predictions.pivot_table(
    index=["path", "true_label"],   # rows by recordingo
    columns="concept_name",      # each concept becomes its own column
    values="magnitude"          # the numeric values go in
).reset_index(drop = False)

In [13]:
vector_df

concept_name,path,true_label,long_constant_thick,long_dropping_flat_thick,long_dropping_steep_thick,long_dropping_steep_thin,long_rising_flat_thick,long_rising_steep_thick,long_rising_steep_thin,short_constant_thick,short_dropping_steep_thick,short_dropping_steep_thin,short_rising_steep_thick,short_rising_steep_thin
0,RAVDESS\original_data\Actor_01\03-01-01-01-01-...,5,-0.446786,-1.726427,-0.424797,-1.230913,-0.055599,-1.325301,-1.175653,-0.371397,-0.857731,-1.254580,-0.928992,0.077504
1,RAVDESS\original_data\Actor_01\03-01-01-01-02-...,5,-0.686852,-2.365001,-0.016568,-1.471219,0.366648,-1.863038,-1.228735,0.062346,-1.008595,-1.665167,-1.542318,0.371737
2,RAVDESS\original_data\Actor_01\03-01-01-01-02-...,5,0.058950,-1.018274,-0.156854,-0.511465,0.399225,-0.300722,-0.671590,-0.119173,-0.640000,-0.739923,-0.062474,0.228532
3,RAVDESS\original_data\Actor_01\03-01-02-01-01-...,1,-0.086732,-1.201567,0.135671,-0.674904,0.301109,-0.798481,0.353926,-0.327486,0.144517,-0.557534,-0.267235,0.392133
4,RAVDESS\original_data\Actor_01\03-01-02-01-01-...,1,-1.179929,-2.255814,-0.504839,-1.873528,-0.424320,-2.069821,-1.129922,-0.693433,-0.739755,-1.239145,-1.470185,0.006896
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1329,RAVDESS\original_data\Actor_24\03-01-08-01-02-...,7,2.127764,0.941703,1.533425,1.578726,1.506175,1.252627,1.318606,0.860395,0.440154,0.738477,1.597433,0.822107
1330,RAVDESS\original_data\Actor_24\03-01-08-01-02-...,7,2.494300,1.825770,2.021223,2.226799,1.920461,2.048605,2.136045,1.029077,1.033162,1.087814,2.624187,1.409729
1331,RAVDESS\original_data\Actor_24\03-01-08-02-01-...,7,2.354847,1.446679,1.402514,1.777966,1.513534,2.015508,1.992022,0.978621,0.892474,1.132352,2.334282,1.171509
1332,RAVDESS\original_data\Actor_24\03-01-08-02-02-...,7,2.596690,1.976899,1.404304,2.276531,1.693202,2.797426,2.210808,1.157644,1.113487,1.382582,3.190612,1.543425


In [14]:
# Features (X) and labels (y)
X = vector_df.drop(columns=["path", "true_label"], inplace=False).to_numpy()
y = vector_df["true_label"].to_numpy()

In [15]:
X.shape, y.shape

((1334, 12), (1334,))

In [16]:
# added by Noam as KMeans has a known bug when on Windows using MKL with multithreading.
import os
os.environ["OMP_NUM_THREADS"] = "6"
# end of addition

In [17]:
def intra_inter_scores(X, y, metric="euclidean"):
    """Compute intra- and inter-cluster distance statistics for labels y using the given metric.

    Returns:
      - intra_mean: mean of within-class pairwise distances
      - inter_mean: mean of between-class pairwise distances
      - separation: inter_mean / intra_mean
    """
    # 1) Compute pairwise distance matrix for the chosen metric
    dist_matrix = pairwise_distances(X, metric=metric)
    # 2) Unique label identifiers present in y
    labels = np.unique(y)

    # 3) Accumulators for intra- and inter-class distances
    intra_dists = []
    inter_dists = []

    # 4) Intra-class distances: average of upper triangle (exclude diagonal)
    for label in labels:
        idx = np.where(y == label)[0]
        if len(idx) > 1:
            d = dist_matrix[np.ix_(idx, idx)]
            intra_dists.append(d[np.triu_indices_from(d, k=1)].mean())

    # 5) Inter-class distances: mean distances across class pairs
    for i, lbl1 in enumerate(labels):
        for lbl2 in labels[i+1:]:
            idx1 = np.where(y == lbl1)[0]
            idx2 = np.where(y == lbl2)[0]
            d = dist_matrix[np.ix_(idx1, idx2)]
            inter_dists.append(d.mean())

    # 6) Aggregate metrics
    return {
        "intra_mean": np.mean(intra_dists),
        "inter_mean": np.mean(inter_dists),
        "separation": np.mean(inter_dists) / np.mean(intra_dists)
    }

### Explanation: Intra-class distances (Step 4)

- `# 4) Intra-class distances: average of upper triangle (exclude diagonal)`
  - We will compute, for each label, the average pairwise distance between all points that share that label. We use the upper triangle of the submatrix to avoid double-counting pairs and to exclude the diagonal zeros.
- `for label in labels:`
  - Iterate over each unique label present in `y`.
- `idx = np.where(y == label)[0]`
  - Find the indices of samples that belong to the current label.
- `if len(idx) > 1:`
  - Only compute an average if there are at least two samples in this label (otherwise there are no pairs).
- `d = dist_matrix[np.ix_(idx, idx)]`
  - Slice the full pairwise distance matrix to get the within-class submatrix for these indices.
- `intra_dists.append(d[np.triu_indices_from(d, k=1)].mean())`
  - Take the upper-triangular values of `d` (excluding the diagonal) and append their mean to `intra_dists`.

Numerical example to track:
- Let `X = [[0,0], [1,0], [4,0], [4,3], [7,0]]` and `y = [0, 0, 1, 1, 1]`.
- Euclidean pairwise distances (rounded):
  - Between indices (0,1,2,3,4) -> the within-class values are:
    - Label 0 (indices [0,1]): submatrix [[0, 1], [1, 0]] -> upper-triangle values [1] -> mean 1.000.
    - Label 1 (indices [2,3,4]): pairwise distances [3.000, 3.000, 4.243] -> mean ≈ 3.414.
- So `intra_dists = [1.000, 3.414]`, and the overall intra-class mean used later is the average of these: ≈ 2.207.

In [ ]:
# Numeric tracking example for the intra-class computation
import numpy as np
from sklearn.metrics import pairwise_distances

X = np.array([[0,0],[1,0],[4,0],[4,3],[7,0]], dtype=float)
y = np.array([0,0,1,1,1])

dist_matrix = pairwise_distances(X, metric="euclidean")
print("Distance matrix (rounded):\n", np.round(dist_matrix, 3))

labels = np.unique(y)
intra_means = []
for label in labels:
    idx = np.where(y == label)[0]
    print(f"\nLabel {label} indices:", idx.tolist())
    if len(idx) > 1:
        d = dist_matrix[np.ix_(idx, idx)]
        print("Within-class submatrix (rounded):\n", np.round(d, 3))
        tri = d[np.triu_indices_from(d, k=1)]
        print("Upper-triangle (excluding diagonal):", np.round(tri, 3).tolist())
        mean_val = tri.mean()
        intra_means.append(mean_val)
        print("Mean within-class distance:", round(mean_val, 3))

print("\nClass-wise means:", [round(v, 3) for v in intra_means])
print("Overall intra_mean across classes:", round(np.mean(intra_means), 3))

In [33]:
from sklearn.preprocessing import normalize


def evaluate_clustering(X, y, n_clusters, random_state=42, print_results=False):
    # KMeans clustering with Euclidean distance
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    clusters_euc = kmeans.fit_predict(X)
    
    # KMeans clustering with Cosine distance
    X_cosine = normalize(X, norm='l2', axis=1)        # put points on the unit sphere
    clusters_cos = kmeans.fit_predict(X_cosine)
    
    if print_results:
        # ---- STEP 4: Evaluation (since you have ground truth labels) ----
        print("Euclidean clustering ARI:", adjusted_rand_score(y, clusters_euc))
        print("Cosine clustering ARI:", adjusted_rand_score(y, clusters_cos))
        print("Euclidean Silhouette:", silhouette_score(X, clusters_euc))
        print("Cosine Silhouette:", silhouette_score(X_cosine, clusters_cos))
        
        # Intra-cluster and inter-cluster scores
        print("Euclidean:", intra_inter_scores(X, clusters_euc, metric="euclidean"))
        print("Cosine:", intra_inter_scores(X, clusters_cos, metric="cosine"))
    
    # return {
    #             "Euclidean clustering ARI": adjusted_rand_score(y, clusters_euc),
    #             "Cosine clustering ARI": adjusted_rand_score(y, clusters_cos),
    #             "Euclidean Silhouette": silhouette_score(X, clusters_euc),
    #             "Cosine Silhouette": silhouette_score(X_cosine, clusters_cos),
    #             "Euclidean": intra_inter_scores(X, clusters_euc, metric="euclidean"),
    #             "Cosine": intra_inter_scores(X, clusters_cos, metric="cosine")
    #         }

In [41]:
evaluate_clustering(X, y, n_clusters=4, print_results=True)

c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Euclidean clustering ARI: 0.38182743672621716
Cosine clustering ARI: 0.41406291854917626
Euclidean Silhouette: 0.41195577060284433
Cosine Silhouette: 0.44246297408149293
Euclidean: {'intra_mean': 2.5104280795515517, 'inter_mean': 6.258408444023364, 'separation': 2.4929646441579516}
Cosine: {'intra_mean': 0.2186656811988295, 'inter_mean': 1.2271612419480342, 'separation': 5.6120431666283945}


In [ ]:
evaluate_clustering(X, y, n_clusters=5, print_results=True)

c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Euclidean clustering ARI: 0.3882470307116555
Cosine clustering ARI: 0.4564429666539113
Euclidean Silhouette: 0.3811002493763267
Cosine Silhouette: 0.437813592483154
Euclidean: {'intra_mean': 2.2482574383031233, 'inter_mean': 6.074404013965967, 'separation': 2.7018276067844953}
Cosine: {'intra_mean': 0.1962620095654702, 'inter_mean': 1.1109259519523622, 'separation': 5.660422791002622}


In [39]:
evaluate_clustering(X, y, n_clusters=6, print_results=True)

c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Euclidean clustering ARI: 0.3765893405826857
Cosine clustering ARI: 0.4901057960555005
Euclidean Silhouette: 0.32491039038016717
Cosine Silhouette: 0.4310011705168567
Euclidean: {'intra_mean': 2.1572021177804177, 'inter_mean': 5.981022637106708, 'separation': 2.7725833327387446}
Cosine: {'intra_mean': 0.20993053713056165, 'inter_mean': 1.1243053344184557, 'separation': 5.355606429564932}


In [40]:
evaluate_clustering(X, y, n_clusters=8, print_results=True)

c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Euclidean clustering ARI: 0.41341367810854185
Cosine clustering ARI: 0.49099731164710997
Euclidean Silhouette: 0.31102350180162175
Cosine Silhouette: 0.39055751801903504
Euclidean: {'intra_mean': 1.9592480828619923, 'inter_mean': 5.802199997562347, 'separation': 2.961442222817808}
Cosine: {'intra_mean': 0.2043180404923596, 'inter_mean': 1.0342021261012069, 'separation': 5.061726921465266}


## Clustering with good CAVs only

In [20]:
######### filter all bad CAVs #########

# load tcav_per_sample_with_acc.csv
acc_df = pd.read_csv(Path("tcav_per_sample_with_acc.csv"))
acc_df.groupby("concept_name")["cav_acc"].mean()

# mask with only 85% and up remaining true
high_acc_concepts = acc_df[acc_df["cav_acc"] >= 0.85]["concept_name"].unique()
high_acc_concepts

array(['long_constant_thick', 'long_dropping_flat_thick',
       'long_dropping_steep_thin', 'long_rising_steep_thick',
       'short_rising_steep_thick'], dtype=object)

In [21]:
# drop from vector_df any columns not in high_acc_concepts
vector_df_high_acc = vector_df[["path", "true_label"] + list(high_acc_concepts)]
vector_df_high_acc

concept_name,path,true_label,long_constant_thick,long_dropping_flat_thick,long_dropping_steep_thin,long_rising_steep_thick,short_rising_steep_thick
0,RAVDESS\original_data\Actor_01\03-01-01-01-01-...,5,-0.446786,-1.726427,-1.230913,-1.325301,-0.928992
1,RAVDESS\original_data\Actor_01\03-01-01-01-02-...,5,-0.686852,-2.365001,-1.471219,-1.863038,-1.542318
2,RAVDESS\original_data\Actor_01\03-01-01-01-02-...,5,0.058950,-1.018274,-0.511465,-0.300722,-0.062474
3,RAVDESS\original_data\Actor_01\03-01-02-01-01-...,1,-0.086732,-1.201567,-0.674904,-0.798481,-0.267235
4,RAVDESS\original_data\Actor_01\03-01-02-01-01-...,1,-1.179929,-2.255814,-1.873528,-2.069821,-1.470185
...,...,...,...,...,...,...,...
1329,RAVDESS\original_data\Actor_24\03-01-08-01-02-...,7,2.127764,0.941703,1.578726,1.252627,1.597433
1330,RAVDESS\original_data\Actor_24\03-01-08-01-02-...,7,2.494300,1.825770,2.226799,2.048605,2.624187
1331,RAVDESS\original_data\Actor_24\03-01-08-02-01-...,7,2.354847,1.446679,1.777966,2.015508,2.334282
1332,RAVDESS\original_data\Actor_24\03-01-08-02-02-...,7,2.596690,1.976899,2.276531,2.797426,3.190612


In [22]:
# Features (X) and labels (y)
X = vector_df_high_acc.drop(columns=["path", "true_label"], inplace=False).to_numpy()
y = vector_df_high_acc["true_label"].to_numpy()

In [23]:
evaluate_clustering(X, y, n_clusters=len(set(y)), print_results=True)

c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Euclidean clustering ARI: 0.3349275881776101
Cosine clustering ARI: 0.44970003400842135
Euclidean Silhouette: 0.32942035951679277
Cosine Silhouette: 0.41180696361392893
Euclidean: {'intra_mean': 1.3360902785344662, 'inter_mean': 4.550672341633388, 'separation': 3.405961718862995}
Cosine: {'intra_mean': 0.1833240719175856, 'inter_mean': 1.1084983093025504, 'separation': 6.04665987236462}


## with features normalized

In [28]:
from sklearn.preprocessing import StandardScaler, normalize

# Features (X) and labels (y)
X = vector_df.drop(columns=["path", "true_label"], inplace=False).to_numpy()
y = vector_df["true_label"].to_numpy()

X_std = StandardScaler().fit_transform(X)   # z-score per feature

In [29]:
evaluate_clustering(X_std, y, n_clusters=len(set(y)), print_results=True)

c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
c:\Users\noams\anaconda3\envs\tf-TTS\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


Euclidean clustering ARI: 0.4292173565866831
Cosine clustering ARI: 0.49998467757527393
Euclidean Silhouette: 0.3023781892733433
Cosine Silhouette: 0.3693857695272717
Euclidean: {'intra_mean': 1.8052613746008412, 'inter_mean': 4.814346964654878, 'separation': 2.66684206087297}
Cosine: {'intra_mean': 0.20958331396236102, 'inter_mean': 1.0540746578320426, 'separation': 5.029382530048854}


# Centroid per Label Clustering